# Final Threshold Policy Stress Audit

Ce notebook reprend le script `final_threshold_policy_stress_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Stress-test des seuils fixes, indispensable pour une alarme live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Stress-test fixed final-score threshold policies.
- Commande de reproduction referencee : final threshold policy stress audit.
- Artefacts controles : Fixed global-threshold policy stress audit exists. (`runs/exp_061_final_threshold_policy_stress/metrics/window_fixed_threshold_summary.csv`).
- Run par defaut : `runs/exp_061_final_threshold_policy_stress`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_threshold_policy_stress_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, make_run_dir


WINDOW_SCORE_VARIANTS = [
    "final_learned_meta_mean",
    "final_attention_ppe_prior",
    "final_safety_sensitive_rule",
    "final_sequence_only",
]

SUBCLIP_SCORE_VARIANTS = [
    "final_learned_meta_mean",
    "final_attention_ppe_prior",
    "final_safety_sensitive_rule",
    "final_sequence_only",
]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `numeric`

Cette cellule definit `numeric`. Elle prepare une partie du script.

In [ ]:
def numeric(df, cols):
    out = df.copy()
    for col in cols:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


## Fonction `select_window_threshold`

Cette cellule definit `select_window_threshold`. Elle prepare une partie du script.

In [ ]:
def select_window_threshold(val_mean, policy):
    work = val_mean.copy()
    work = numeric(
        work,
        [
            "threshold",
            "window_precision",
            "window_recall",
            "window_f1",
            "danger_clip_hit_rate",
            "safe_false_alarms_per_min",
            "median_early_warning_s",
        ],
    )
    if policy == "balanced_window":
        work["policy_score"] = (
            work["window_f1"].fillna(0)
            + 0.50 * work["danger_clip_hit_rate"].fillna(0)
            + 0.15 * work["window_precision"].fillna(0)
            - 0.03 * work["safe_false_alarms_per_min"].fillna(20).clip(upper=20)
        )
        return work.sort_values(["policy_score", "danger_clip_hit_rate", "safe_false_alarms_per_min"], ascending=[False, False, True]).iloc[0]
    if policy == "early_high_recall":
        eligible = work[(work["median_early_warning_s"] >= 1.0) & (work["danger_clip_hit_rate"] >= 0.85)]
        if eligible.empty:
            eligible = work[work["median_early_warning_s"] >= 1.0]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["danger_clip_hit_rate", "median_early_warning_s", "safe_false_alarms_per_min", "window_precision"], ascending=[False, False, True, False]).iloc[0]
    if policy == "low_false_alarm":
        eligible = work[work["safe_false_alarms_per_min"] <= 3.0]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["danger_clip_hit_rate", "window_f1", "window_precision", "safe_false_alarms_per_min"], ascending=[False, False, False, True]).iloc[0]
    if policy == "precision_guard":
        eligible = work[work["window_precision"] >= 0.50]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["danger_clip_hit_rate", "window_f1", "safe_false_alarms_per_min", "window_precision"], ascending=[False, False, True, False]).iloc[0]
    raise ValueError(policy)


## Fonction `select_subclip_threshold`

Cette cellule definit `select_subclip_threshold`. Elle prepare une partie du script.

In [ ]:
def select_subclip_threshold(val_mean, policy):
    work = val_mean.copy()
    work = numeric(work, ["threshold", "precision", "recall", "f1", "balanced_accuracy", "false_alarms_per_min"])
    if policy == "balanced_f1":
        return work.sort_values(["f1", "balanced_accuracy", "recall", "false_alarms_per_min"], ascending=[False, False, False, True]).iloc[0]
    if policy == "high_recall":
        eligible = work[work["recall"] >= 0.90]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["recall", "f1", "false_alarms_per_min", "precision"], ascending=[False, False, True, False]).iloc[0]
    if policy == "low_false_alarm":
        eligible = work[work["false_alarms_per_min"] <= 5.0]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["recall", "f1", "precision", "false_alarms_per_min"], ascending=[False, False, False, True]).iloc[0]
    raise ValueError(policy)


## Fonction `nearest_rows`

Cette cellule definit `nearest_rows`. Elle prepare une partie du script.

In [ ]:
def nearest_rows(df, threshold):
    work = df.copy()
    work["threshold_distance"] = (pd.to_numeric(work["threshold"], errors="coerce") - float(threshold)).abs()
    idx = work.groupby("repeat_seed")["threshold_distance"].idxmin()
    return work.loc[idx].copy()


## Fonction `summarize_repeat_rows`

Cette cellule definit `summarize_repeat_rows`. Elle prepare une partie du script.

In [ ]:
def summarize_repeat_rows(rows, group_cols, metric_cols):
    out = []
    for keys, group in rows.groupby(group_cols):
        key_tuple = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols, key_tuple))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        for col in metric_cols:
            values = pd.to_numeric(group[col], errors="coerce")
            row[f"{col}_mean"] = float(values.mean())
            row[f"{col}_std"] = float(values.std(ddof=0))
            row[f"{col}_min"] = float(values.min())
            row[f"{col}_max"] = float(values.max())
        out.append(row)
    return pd.DataFrame(out)


## Fonction `per_seed_threshold_std`

Cette cellule definit `per_seed_threshold_std`. Elle prepare une partie du script.

In [ ]:
def per_seed_threshold_std(sweep, selector, policy, split_filter):
    selected = []
    for seed, group in sweep.groupby("repeat_seed"):
        val = split_filter(group)
        if val.empty:
            continue
        selected.append({"repeat_seed": seed, "threshold": float(selector(val, policy)["threshold"])})
    if not selected:
        return np.nan, np.nan
    values = pd.DataFrame(selected)["threshold"].to_numpy(dtype=np.float64)
    return float(values.mean()), float(values.std(ddof=0))


## Fonction `window_policy_audit`

Cette cellule definit `window_policy_audit`. Elle prepare une partie du script.

In [ ]:
def window_policy_audit(sweeps):
    sweeps = sweeps[sweeps["score_variant"].isin(WINDOW_SCORE_VARIANTS)].copy()
    sweeps = numeric(
        sweeps,
        [
            "threshold",
            "window_precision",
            "window_recall",
            "window_f1",
            "danger_clip_hit_rate",
            "safe_false_alarms_per_min",
            "median_early_warning_s",
        ],
    )
    detail_rows = []
    policies = ["balanced_window", "early_high_recall", "low_false_alarm", "precision_guard"]
    for score in WINDOW_SCORE_VARIANTS:
        score_df = sweeps[sweeps["score_variant"].eq(score)].copy()
        for policy in policies:
            val_mean = (
                score_df[score_df["split"].eq("val")]
                .groupby("threshold", as_index=False)[
                    ["window_precision", "window_recall", "window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min", "median_early_warning_s"]
                ]
                .mean(numeric_only=True)
            )
            if val_mean.empty:
                continue
            selected = select_window_threshold(val_mean, policy)
            threshold = float(selected["threshold"])
            test_nearest = nearest_rows(score_df[score_df["split"].eq("test")], threshold)
            per_seed_mean, per_seed_std = per_seed_threshold_std(
                score_df,
                select_window_threshold,
                policy,
                lambda group: (
                    group[group["split"].eq("val")]
                    .groupby("threshold", as_index=False)[
                        ["window_precision", "window_recall", "window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min", "median_early_warning_s"]
                    ]
                    .mean(numeric_only=True)
                ),
            )
            for _, row in test_nearest.iterrows():
                item = row.to_dict()
                item["policy"] = policy
                item["global_threshold_from_val"] = threshold
                item["per_seed_selected_threshold_mean"] = per_seed_mean
                item["per_seed_selected_threshold_std"] = per_seed_std
                detail_rows.append(item)
    detail = pd.DataFrame(detail_rows)
    summary = summarize_repeat_rows(
        detail,
        ["score_variant", "policy"],
        [
            "global_threshold_from_val",
            "per_seed_selected_threshold_std",
            "window_precision",
            "window_recall",
            "window_f1",
            "danger_clip_hit_rate",
            "safe_false_alarms_per_min",
            "median_early_warning_s",
            "late_or_missed_clips",
        ],
    )
    summary["meets_hit_0.85"] = summary["danger_clip_hit_rate_mean"] >= 0.85
    summary["meets_fa_le_5"] = summary["safe_false_alarms_per_min_mean"] <= 5.0
    summary["meets_early_1s"] = summary["median_early_warning_s_mean"] >= 1.0
    summary["meets_precision_0.40"] = summary["window_precision_mean"] >= 0.40
    summary["deployment_gate_count"] = (
        summary["meets_hit_0.85"].astype(int)
        + summary["meets_fa_le_5"].astype(int)
        + summary["meets_early_1s"].astype(int)
        + summary["meets_precision_0.40"].astype(int)
    )
    return detail, summary


## Fonction `subclip_policy_audit`

Cette cellule definit `subclip_policy_audit`. Elle prepare une partie du script.

In [ ]:
def subclip_policy_audit(sweeps):
    sweeps = sweeps[
        sweeps["score_variant"].isin(SUBCLIP_SCORE_VARIANTS)
        & sweeps["aggregation"].eq("risk_max")
        & sweeps["subclip_bin_s"].astype(float).eq(3.0)
    ].copy()
    sweeps = numeric(sweeps, ["threshold", "precision", "recall", "f1", "balanced_accuracy", "false_alarms_per_min"])
    detail_rows = []
    policies = ["balanced_f1", "high_recall", "low_false_alarm"]
    for score in SUBCLIP_SCORE_VARIANTS:
        score_df = sweeps[sweeps["score_variant"].eq(score)].copy()
        for policy in policies:
            val_mean = (
                score_df[score_df["split"].eq("val")]
                .groupby("threshold", as_index=False)[["precision", "recall", "f1", "balanced_accuracy", "false_alarms_per_min"]]
                .mean(numeric_only=True)
            )
            if val_mean.empty:
                continue
            selected = select_subclip_threshold(val_mean, policy)
            threshold = float(selected["threshold"])
            test_nearest = nearest_rows(score_df[score_df["split"].eq("test")], threshold)
            per_seed_mean, per_seed_std = per_seed_threshold_std(
                score_df,
                select_subclip_threshold,
                policy,
                lambda group: (
                    group[group["split"].eq("val")]
                    .groupby("threshold", as_index=False)[["precision", "recall", "f1", "balanced_accuracy", "false_alarms_per_min"]]
                    .mean(numeric_only=True)
                ),
            )
            for _, row in test_nearest.iterrows():
                item = row.to_dict()
                item["policy"] = policy
                item["global_threshold_from_val"] = threshold
                item["per_seed_selected_threshold_mean"] = per_seed_mean
                item["per_seed_selected_threshold_std"] = per_seed_std
                detail_rows.append(item)
    detail = pd.DataFrame(detail_rows)
    summary = summarize_repeat_rows(
        detail,
        ["score_variant", "policy"],
        [
            "global_threshold_from_val",
            "per_seed_selected_threshold_std",
            "precision",
            "recall",
            "f1",
            "balanced_accuracy",
            "false_alarms_per_min",
        ],
    )
    summary["meets_recall_0.90"] = summary["recall_mean"] >= 0.90
    summary["meets_fa_le_8"] = summary["false_alarms_per_min_mean"] <= 8.0
    summary["meets_precision_0.50"] = summary["precision_mean"] >= 0.50
    summary["deployment_gate_count"] = (
        summary["meets_recall_0.90"].astype(int)
        + summary["meets_fa_le_8"].astype(int)
        + summary["meets_precision_0.50"].astype(int)
    )
    return detail, summary


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, window_summary, subclip_summary):
    lines = ["# Final Threshold Policy Stress Audit", ""]
    lines.append("This audit selects one global threshold from validation-repeat means and applies it unchanged to test repeats. It is stricter than choosing a separate threshold per repeat and is closer to a real deployment policy.")
    lines.append("")
    lines.append("## Window-Level Fixed Thresholds")
    lines.append("")
    lines.append("| score | policy | threshold | hit | FA/min | precision | F1 | median early s | threshold std | gates |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    view = window_summary.copy()
    view["rank"] = (
        view["deployment_gate_count"]
        + 0.40 * view["danger_clip_hit_rate_mean"].fillna(0)
        + 0.20 * view["window_f1_mean"].fillna(0)
        + 0.10 * view["window_precision_mean"].fillna(0)
        - 0.04 * view["safe_false_alarms_per_min_mean"].fillna(20).clip(upper=20)
    )
    for _, row in view.sort_values("rank", ascending=False).head(16).iterrows():
        lines.append(
            f"| {row['score_variant']} | {row['policy']} | {row['global_threshold_from_val_mean']:.2f} | "
            f"{row['danger_clip_hit_rate_mean']:.3f} | {row['safe_false_alarms_per_min_mean']:.3f} | "
            f"{row['window_precision_mean']:.3f} | {row['window_f1_mean']:.3f} | "
            f"{row['median_early_warning_s_mean']:.3f} | {row['per_seed_selected_threshold_std_mean']:.3f} | "
            f"{int(row['deployment_gate_count'])}/4 |"
        )
    lines.append("")
    lines.append("## 3s Subclip Fixed Thresholds")
    lines.append("")
    lines.append("| score | policy | threshold | recall | FA/min | precision | F1 | balanced acc | threshold std | gates |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    sub = subclip_summary.copy()
    sub["rank"] = (
        sub["deployment_gate_count"]
        + 0.45 * sub["recall_mean"].fillna(0)
        + 0.25 * sub["f1_mean"].fillna(0)
        + 0.15 * sub["precision_mean"].fillna(0)
        - 0.03 * sub["false_alarms_per_min_mean"].fillna(20).clip(upper=20)
    )
    for _, row in sub.sort_values("rank", ascending=False).head(16).iterrows():
        lines.append(
            f"| {row['score_variant']} | {row['policy']} | {row['global_threshold_from_val_mean']:.2f} | "
            f"{row['recall_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | "
            f"{row['precision_mean']:.3f} | {row['f1_mean']:.3f} | {row['balanced_accuracy_mean']:.3f} | "
            f"{row['per_seed_selected_threshold_std_mean']:.3f} | {int(row['deployment_gate_count'])}/3 |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- A single universal threshold is not justified by the data; the correct result is a small set of named operating policies.")
    lines.append("- The transparent `final_attention_ppe_prior` and learned `final_learned_meta_mean` remain the most defensible balanced score families.")
    lines.append("- High-sensitivity policies increase recall/early warning but have a visible false-alarm and precision cost.")
    lines.append("- Threshold finality remains a policy decision, but it is now bounded by fixed-threshold validation-to-test evidence rather than informal preference.")
    summary_path = run_dir / "final_threshold_policy_stress_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Final Threshold Policy Stress Audit", f"- Summary: `{summary_path}`")
    return summary_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    final_score_run = resolve(args.final_score_run)
    subclip_run = resolve(args.subclip_run)
    window_sweeps = pd.read_csv(final_score_run / "metrics" / "final_score_threshold_sweeps.csv")
    subclip_sweeps = pd.read_csv(subclip_run / "metrics" / "final_score_subclip_threshold_sweeps.csv")
    window_detail, window_summary = window_policy_audit(window_sweeps)
    subclip_detail, subclip_summary = subclip_policy_audit(subclip_sweeps)
    window_detail.to_csv(run_dir / "metrics" / "window_fixed_threshold_details.csv", index=False)
    window_summary.to_csv(run_dir / "metrics" / "window_fixed_threshold_summary.csv", index=False)
    subclip_detail.to_csv(run_dir / "metrics" / "subclip_fixed_threshold_details.csv", index=False)
    subclip_summary.to_csv(run_dir / "metrics" / "subclip_fixed_threshold_summary.csv", index=False)
    write_json(
        run_dir / "metrics" / "threshold_policy_stress_config.json",
        {
            "final_score_run": str(final_score_run),
            "subclip_run": str(subclip_run),
            "window_policy": "global threshold selected from validation mean across repeat seeds and applied to test repeat seeds",
            "subclip_policy": "3s risk_max subclips; global threshold selected from validation mean across repeat seeds and applied to test repeat seeds",
            "window_gates": ["hit>=0.85", "FA/min<=5", "median early>=1s", "precision>=0.40"],
            "subclip_gates": ["recall>=0.90", "FA/min<=8", "precision>=0.50"],
        },
    )
    summary_path = write_summary(run_dir, window_summary, subclip_summary)
    print(run_dir)
    print(summary_path)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Stress-test fixed final-score threshold policies.")
    parser.add_argument("--run-name", default="exp_061_final_threshold_policy_stress")
    parser.add_argument("--final-score-run", default="runs/exp_039_final_aggregated_score")
    parser.add_argument("--subclip-run", default="runs/exp_047_final_aggregated_subclip_eval")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_061_final_threshold_policy_stress_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["final_threshold_policy_stress_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
